# 🏆 Agency Cup: start here

**Case #002: The Phantom Leak.** A secret file left the company. Your team builds an AI agent that finds out who did it.

In this notebook you will:

1. check that your laptop is ready,
2. meet the game's **MCP server** and try its tools by hand,
3. run a starter detective agent that uses those MCP tools,
4. make it better, and
5. submit your report.

> ⚠️ **Three things to know before you run anything**
> - **Every tool call costs budget points**, even from this notebook. Each case has a fixed budget, and it does not refill.
> - **You get 3 submissions per case.** Each extra one costs 8 points.
> - **Learn in the training round.** Save the qualifier and the final for your best agent.

## Step 0: Check your setup

Do this once, in a terminal, in this folder:

```bash
uv sync
```

Then copy `.env.example` to `.env` and fill in `TEAM_KEY` and `NVIDIA_API_KEY`.

In VS Code or Cursor, click **Select Kernel** (top right) and choose **`.venv`**. Then run the cell below. Every line must show ✅.

In [ ]:
from check_setup import check_mcp, check_model, check_setup

if check_setup():
    await check_mcp()
    await check_model()

## Step 1: Meet the MCP server

**MCP** (Model Context Protocol) is a standard way to give tools to an agent. Think of it as a USB port for tools.

- The game is an **MCP server**. It owns the tools.
- Your agent is an **MCP client**. It connects and asks: *"Which tools do you have?"*
- The server answers with each tool's **name**, **description** and **inputs**. The agent reads that list and decides what to call.

So you write **no tool code**. Any framework that speaks MCP can plug in. The cell below asks the question your agent will ask. It is free.

In [ ]:
from agency_kit.case_mcp import call_tool, list_tools, mcp_url

print("MCP server:", mcp_url(), "\n")
for t in await list_tools():
    print(f"🔧 {t['name']}({', '.join(t['inputs'])})")
    print(f"     {t['description']}")

🗣️ **Talk in your team:** the model only sees these names and descriptions. Read them the way your agent will.

Now call a tool the way your agent does. `case_brief` is free.

In [ ]:
brief = await call_tool("case_brief")
print(brief.get("brief") or brief)

In [ ]:
# What each tool costs, and how much budget you have left. Both calls are free.
print("Costs:", brief.get("costs"))
print("Status:", await call_tool("status"))

## Step 2: Try the tools by hand

Every call below goes through MCP, exactly like your agent's calls. Look before you spend. `list_sources` is free. `search` costs 1 point and returns at most 5 documents. `open_document` costs 1 point the first time.

🗣️ **Talk in your team:** which source would you check first, and why?

In [ ]:
for s in (await call_tool("list_sources"))["sources"]:
    print(f"{s['source']:<12} {s['documents']:>3} documents   {s['about']}")

In [ ]:
QUERY = "deck"   # ✏️ change me: try a name, a device or a file from the brief
SOURCE = ""      # ✏️ optional: one source from the list above, for example "dlp"

found = await call_tool("search", query=QUERY, source=SOURCE)
print(f"cost {found.get('cost')} · budget left {found.get('budget_left', '?')}")
for hit in found.get("results", []):
    print(hit)

In [ ]:
hits = found.get("results", [])
DOC_ID = hits[0]["id"] if hits else ""   # ✏️ or paste another ID, for example "EV-7Q2K"

if DOC_ID:
    doc = await call_tool("open_document", doc_id=DOC_ID)
    print(doc.get("title"))
    print()
    print(doc.get("body") or doc)
else:
    print("No results. Change QUERY above and run both cells again.")

## Step 3: Run the starter detective

This agent uses the **OpenAI Agents SDK**, your **NVIDIA model** and the game's **MCP tools**. The code is in `starters/agent_openai_sdk.py`.

Look at how little code connects the agent to the game. There is no tool code at all:

```python
async with MCPServerStreamableHttp(params={"url": mcp_url(), "headers": mcp_headers()}) as case_tools:
    detective = Agent(name="Detective", instructions=instructions, model=model, mcp_servers=[case_tools])
```

It runs as a **dry run**: the agent investigates but cannot submit, so you do not use a submission. It still spends budget points. It takes a few minutes, because the client stays under the NVIDIA rate limit.

> The starter is bare on purpose: it has almost no instructions and no plan. Turning it into a detective is your team's job.

In [ ]:
from starters.agent_openai_sdk import INSTRUCTIONS, run_detective, show_steps

result = await run_detective()   # dry run: no submission
show_steps(result)
print("\n=== The detective's answer ===\n")
print(result.final_output)

In [ ]:
print(await call_tool("status"))   # how much budget did that run use?

## Step 4: Make it better

Read the steps above like a detective. Where did the agent go wrong?

What did it spend budget on? What did it miss in the brief? How to fix that is your team's work: this kit gives no strategy.

Write your own rules below and run again. Each run spends budget, so change one thing at a time.

In [ ]:
MY_INSTRUCTIONS = INSTRUCTIONS + """
Extra rules from our team:
- ✏️ Write your own rules here.
"""

RUN_AGAIN = False   # ✏️ set to True when your new instructions are ready

if RUN_AGAIN:
    result = await run_detective(instructions=MY_INSTRUCTIONS)
    show_steps(result)
    print("\n=== The detective's answer ===\n")
    print(result.final_output)

## Step 5: Submit your report

You can let the agent submit, or submit by hand. Either way, **only documents your team opened count as evidence**.

**Option A: the agent submits.** This is what counts in the qualifier and the final.

In [ ]:
AGENT_SUBMITS = False   # ✏️ set to True to let the agent call submit_report

if AGENT_SUBMITS:
    result = await run_detective(instructions=MY_INSTRUCTIONS, submit=True)
    show_steps(result)
    print(result.final_output)

**Option B: submit by hand.** Useful in the training round, to see how scoring feels.

In [ ]:
REPORT = dict(
    culprit="",          # a username from the brief
    accomplice="",       # the username of the person whose account was borrowed
    channel="",          # printer | usb | personal_email | cloud_link
    time_window="",      # "HH:MM" in LOCAL office time, start of the 15-minute slot
    motive="",           # money | new_job | revenge | ethics | blackmail
    evidence=[],         # 3 to 8 document IDs that you opened
)
SUBMIT_NOW = False       # ✏️ set to True only when you are sure

if SUBMIT_NOW:
    print(await call_tool("submit_report", **REPORT))
else:
    print("Not submitted. Fill in REPORT and set SUBMIT_NOW = True.")

## What next?

- Watch your team's moves live on the **My team** page of the game website, and your rank on the **Leaderboard**.
- Prefer another framework? Run `uv run python starters/agent_langgraph.py` (LangGraph) or `uv run python starters/agent_adk.py` (Google ADK). Both plug into the same MCP server, with no tool code.
- Run from a terminal: `uv run python starters/agent_openai_sdk.py` (dry run) or add `--submit`.
- The rules and the scoring are in `README.md`.

Good luck, detectives. 🕵️